In [24]:
import pandas as pd
import os

### load the "master_orders.csv"

In [25]:
df = pd.read_csv(os.path.join("../Merged_Olist_Data", "master_orders.csv"))
print(df.shape)
df.head()

(99441, 18)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


### The date columns in our CSV are stored as plain text strings like "2017-10-02 10:56:33".converted them into real datetime objects.


In [26]:
# Convert ALL date columns in one go
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(df[date_columns].dtypes)

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [27]:
print(df.dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_id                                   str
review_score                            float64
review_comment_title                        str
review_comment_message                      str
review_creation_date                        str
review_answer_timestamp                     str
customer_unique_id                          str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
dtype: object


### Calculate Days_Difference:
Days_Difference = Estimated Delivery Date − Actual Delivery Date

In [28]:
df["Days_Difference"] = (
    df["order_estimated_delivery_date"] - df["order_delivered_customer_date"]
).dt.days

print(df["Days_Difference"].describe())

count    96476.000000
mean        10.876881
std         10.183854
min       -189.000000
25%          6.000000
50%         11.000000
75%         16.000000
max        146.000000
Name: Days_Difference, dtype: float64


### Checking for late deliveries to know if everything is working

In [29]:
late_deliveries = df[df["Days_Difference"] < 0][
    ["order_id", "order_estimated_delivery_date",
     "order_delivered_customer_date", "Days_Difference"]
]

print(late_deliveries.head())

                            order_id order_estimated_delivery_date  \
20  203096f03d82e0dffbc41ebc2e2bcfb7                    2017-09-28   
25  fbf9ac61453ac646ce8ad9783d7d0af6                    2018-03-12   
35  8563039e855156e48fccee4d611a3196                    2018-03-20   
41  6ea2f835b4556291ffdc53fa0b3b95e8                    2017-12-21   
57  66e4624ae69e7dc89bd50222b59f581f                    2018-04-02   

   order_delivered_customer_date  Days_Difference  
20           2017-10-09 22:23:46            -12.0  
25           2018-03-21 22:03:54            -10.0  
35           2018-03-20 00:59:25             -1.0  
41           2017-12-28 18:59:23             -8.0  
57           2018-04-03 13:28:46             -2.0  


### Classify Every Order into a Delivery Status:
On Time : Days_Difference >= 0 - Arrived on or before the promised date
Late : Days_Difference between -5 and 0 - Arrived up to 5 days after the promised date
Super Late : Days_Difference < -5 - Arrived more than 5 days after the promised date
Pending : No delivery date recorded - Order has not been delivered yet

In [30]:
def classify_delivery(row):
    if pd.isnull(row["order_delivered_customer_date"]):
        return "Pending"
    elif row["Days_Difference"] >= 0:
        return "On Time"
    elif row["Days_Difference"] >= -5:
        return "Late"
    else:
        return "Super Late"

df["Delivery_Status"] = df.apply(classify_delivery, axis=1)

print(df["Delivery_Status"].value_counts())

Delivery_Status
On Time       88649
Super Late     4212
Late           3615
Pending        2965
Name: count, dtype: int64


### Removing Cancelled & Unavailable Orders

In [31]:
df_cancelled = df[df["order_status"].isin(["canceled", "unavailable"])]
print(f"Cancelled/Unavailable orders removed: {len(df_cancelled)}")

df = df[~df["order_status"].isin(["canceled", "unavailable"])]
print(f"Remaining orders for analysis: {len(df)}")

Cancelled/Unavailable orders removed: 1234
Remaining orders for analysis: 98207


In [32]:
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,Days_Difference,Delivery_Status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,7.0,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,5.0,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,17.0,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03 00:00:00,2017-12-05 19:21:58,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,12.0,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17 00:00:00,2018-02-18 13:02:51,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,9.0,On Time


### Overall Delivery Performance Summary

In [33]:
Total = len(df)
summary = df["Delivery_Status"].value_counts()

print("=" * 40)
print("VERIDI LOGISTICS — DELIVERY PERFORMANCE")
print("=" * 40)
for status, count in summary.items():
    print(f"{status:12}: {count:6} orders ({count/Total*100:.1f}%)")
print("=" * 40)
print(f"{'TOTAL':12}: {Total:6} orders")

VERIDI LOGISTICS — DELIVERY PERFORMANCE
On Time     :  88644 orders (90.3%)
Super Late  :   4211 orders (4.3%)
Late        :   3615 orders (3.7%)
Pending     :   1737 orders (1.8%)
TOTAL       :  98207 orders


In [34]:
output_path = os.path.join("../Merged_Olist_Data", "delivery_analysis.csv")
df.to_csv(output_path, index=False)
print(f"delivery_analysis.csv saved to {output_path}")

delivery_analysis.csv saved to Merged_Olist_Data\delivery_analysis.csv
